# Notebook 11 — state dependence

Level-1 code: the matrix at three temperatures, at full resolution (one group per line) so that only the state matters; can $R(T_1)$ predict $T_2$? Then interpolate.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom
from rtedu.transport import run, emergent_by_line
from rtedu.matrix import group_of_line, MacroatomRedistribution, build_R, MatrixRedistribution, low_rank, interpolate_R, row_error
from rtedu.redistribution import EpsilonRedistribution
from rtedu.bands import band_fluxes, magnitudes, colours
rng = np.random.default_rng(rtedu.SEEDS["ch11"])
atom = five_level_atom(); nm = 1e7 * atom.lam_cm
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nu_launch = atom.nu[0] * 1.001
def spectrum(model, n, seed):
    """emergent line spectrum, band magnitudes and colours of n blue packets under a redistribution model"""
    nu, last, n_int = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau, r_out, t, model)
    m = magnitudes(band_fluxes(nu)); return emergent_by_line(last, atom.n_lines), m, colours(m), float(n_int.mean())

In [ ]:
def R_at(T_, n_g=10, n=6000, seed=0):
    tau_ = atom.line_list(T_, n_total, t); emis_ = atom.thermal_emissivity(T_, n_total)
    macro = MacroatomRedistribution(atom, tau_)
    nu, last, _ = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau_, r_out, t, macro)
    g, ng = group_of_line(atom.nu, n_g)
    usage = np.bincount([g[k] for k, _ in macro.events], minlength=ng)      # how often each row is used
    return build_R(macro.events, g, ng), emergent_by_line(last, atom.n_lines), tau_, emis_, usage

Ts = [2500.0, 4000.0, 7000.0]
tab = {T_: R_at(T_, seed=rtedu.SEEDS["ch11"] + i) for i, T_ in enumerate(Ts)}
for T_ in Ts:
    print(f"T = {T_:.0f} K: tau range {tab[T_][2].min():.3f} .. {tab[T_][2].max():.1f}; events per row {tab[T_][4]}")
usage = tab[4000.0][4]
print("the reddest line's row takes", f"{usage[0] / usage.sum():.3f}", "of all absorptions: which rows matter is set by the transport, not by the atom")

## Using the wrong state

Transport at $T_2$ with $R(T_1)$, $R(T_2)$ and $R(T_3)$, and with the interpolated $R$ from the two neighbours.

In [ ]:
n = 6000; g4, _ = group_of_line(atom.nu, 10)              # ten groups: one per line, so only the state matters
def spectrum_at(T_, model, seed):
    tau_ = atom.line_list(T_, n_total, t)
    nu, last, _ = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau_, r_out, t, model)
    return emergent_by_line(last, atom.n_lines)
T_test = 4000.0; emis_test = atom.thermal_emissivity(T_test, n_total)
ref = tab[T_test][1]; noise = float(4 * np.sqrt(ref * (1 - ref) / n).sum())
errs = {}
for T_ in Ts:
    errs[f"R({T_:.0f})"] = float(np.abs(spectrum_at(T_test, MatrixRedistribution(tab[T_][0], g4, emis_test), rtedu.SEEDS["ch11"] + 20) - ref).sum())
R_int = interpolate_R(T_test, [2500.0, 7000.0], [tab[2500.0][0], tab[7000.0][0]])
errs["R interpolated 2500-7000"] = float(np.abs(spectrum_at(T_test, MatrixRedistribution(R_int, g4, emis_test), rtedu.SEEDS["ch11"] + 21) - ref).sum())
row_err = {f"R({T_:.0f})": row_error(tab[T_][0], tab[T_test][0], usage) for T_ in Ts}; row_err["R interpolated 2500-7000"] = row_error(R_int, tab[T_test][0], usage)   # weighted by row usage
for k in errs: print(f"{k:>26s}: spectrum error {errs[k]:.3f}  usage-weighted row error {row_err[k]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, T_ in zip(axes[:3], Ts):
    im = ax.imshow(tab[T_][0], cmap="viridis", vmin=0, vmax=1); ax.set_title(f"R at {T_:.0f} K (rows: lines by increasing frequency)", fontsize=8); ax.set_xlabel("emitted line"); ax.set_ylabel("absorbed line")
names = list(errs); axes[3].barh(names, [errs[k] for k in names], color=[OI["red"], OI["blue"], OI["red"], OI["green"]]); axes[3].axvline(noise, color="grey", ls="--"); axes[3].set_xlabel("spectrum error at 4000 K"); axes[3].tick_params(axis="y", labelsize=7)
axes[3].set_title("which matrix predicts 4000 K?", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch11_state")

In [ ]:
results.record("ch11", dict(Ts=Ts, T_test=T_test, n=n, n_g=10, R={f"{T_:.0f}": tab[T_][0] for T_ in Ts}, tau_range={f"{T_:.0f}": [float(tab[T_][2].min()), float(tab[T_][2].max())] for T_ in Ts},
                            errs=errs, row_err=row_err, noise=noise, R_interp=R_int, usage=usage, reddest_row_share=float(usage[0] / usage.sum()),
                            reddest_line_nm=float(nm[np.argsort(atom.nu)[0]])))